# displaying attribute

In [ ]:
import pandas as pd

Fire3 = pd.read_csv(r"Data\Fire_Data\viirs-jpss1_2023_Portugal.csv")
Fire2 = pd.read_csv(r"Data\Fire_Data\viirs-jpss1_2022_Portugal.csv")
Fire1 = pd.read_csv(r"Data\Fire_Data\viirs-jpss1_2021_Portugal.csv")

# checking Missing values

In [43]:
Fire3.isna().sum()

latitude      0
longitude     0
bright_ti4    0
scan          0
track         0
acq_date      0
acq_time      0
satellite     0
instrument    0
confidence    0
version       0
bright_ti5    0
frp           0
daynight      0
type          0
dtype: int64

In [44]:
Fire2.isna().sum()

latitude      0
longitude     0
bright_ti4    0
scan          0
track         0
acq_date      0
acq_time      0
satellite     0
instrument    0
confidence    0
version       0
bright_ti5    0
frp           0
daynight      0
type          0
dtype: int64

In [45]:
Fire1.isna().sum()

latitude      0
longitude     0
bright_ti4    0
scan          0
track         0
acq_date      0
acq_time      0
satellite     0
instrument    0
confidence    0
version       0
bright_ti5    0
frp           0
daynight      0
type          0
dtype: int64

# Keeping Fires with high and normal confidence

In [46]:

print(f"fire 2023 shape: {Fire3.shape}")
low = Fire3.shape
Fire3 = Fire3[ Fire3["confidence"].str.lower() != "l" ].copy()
print(F"Dropped {low[0]-Fire3.shape[0]} Fire with low confidence ")

print(f"fire 2022 shape: {Fire2.shape}")
low = Fire2.shape
Fire2 = Fire2[ Fire2["confidence"].str.lower() != "l" ].copy()
print(F"Dropped {low[0]-Fire2.shape[0]} Fire with low confidence ")

print(f"fire 2021 shape: {Fire1.shape}")
low = Fire1.shape
Fire1 = Fire1[ Fire1["confidence"].str.lower() != "l" ].copy()
print(F"Dropped {low[0]-Fire1.shape[0]} Fire with low confidence ")

fire 2023 shape: (3384, 15)
Dropped 169 Fire with low confidence 
fire 2022 shape: (5860, 15)
Dropped 345 Fire with low confidence 
fire 2021 shape: (2501, 15)
Dropped 85 Fire with low confidence 


# Keeping only Vegetation Fire

In [47]:
print(f"fire 2023 shape: {Fire3.shape}")
all = Fire3.shape
Fire3 = Fire3[Fire3["type"] == 0].copy()
print(F"Dropped {all[0]-Fire3.shape[0]} Fire that are'nt presumed vegetation fire ")

print(f"fire 2022 shape: {Fire2.shape}")
all = Fire2.shape
Fire2 = Fire2[Fire2["type"] == 0].copy()
print(F"Dropped {all[0]-Fire2.shape[0]} Fire that are'nt presumed vegetation fire ")    

print(f"fire 2021 shape: {Fire1.shape}")
all = Fire1.shape
Fire1 = Fire1[Fire1["type"] == 0].copy()
print(F"Dropped {all[0]-Fire1.shape[0]} Fire that are'nt presumed vegetation fire ")        

fire 2023 shape: (3215, 15)
Dropped 278 Fire that are'nt presumed vegetation fire 
fire 2022 shape: (5515, 15)
Dropped 207 Fire that are'nt presumed vegetation fire 
fire 2021 shape: (2416, 15)
Dropped 259 Fire that are'nt presumed vegetation fire 


# keeping Fire data from continental Portugal

In [48]:
out = Fire3.shape

Fire3 = Fire3[
    (Fire3['longitude'] > -9.55) & 
    (Fire3['latitude'] > 36.9) & 
    (Fire3['latitude'] < 42.2)
]

print(F"Dropped {out[0]-Fire3.shape[0]} Fire that are'nt located in continental Portugal ")

out = Fire2.shape
Fire2 = Fire2[
    (Fire2['longitude'] > -9.55) & 
    (Fire2['latitude'] > 36.9) & 
    (Fire2['latitude'] < 42.2)
]
print(F"Dropped {out[0]-Fire2.shape[0]} Fire that are'nt located in continental Portugal ")
out = Fire1.shape
Fire1 = Fire1[
    (Fire1['longitude'] > -9.55) &
    (Fire1['latitude'] > 36.9) &
    (Fire1['latitude'] < 42.2)
]
print(F"Dropped {out[0]-Fire1.shape[0]} Fire that are'nt located in continental Portugal ")

Dropped 211 Fire that are'nt located in continental Portugal 
Dropped 26 Fire that are'nt located in continental Portugal 
Dropped 24 Fire that are'nt located in continental Portugal 


In [49]:
cols_to_keep = ['longitude','latitude']
Fire3 = Fire3[cols_to_keep].copy()
Fire2 = Fire2[cols_to_keep].copy()
Fire1 = Fire1[cols_to_keep].copy()

In [ ]:
grid = pd.read_csv(r"Data\csv_final\grid_portugal_1km.csv")

In [51]:
grid.columns

Index(['latitude', 'longitude', 'landcover'], dtype='object')

In [52]:
import numpy as np
from sklearn.neighbors import BallTree

In [53]:
def insert_fire_grid(fire_df,grid_df,column_name):

 all_rad = np.radians(grid_df[["latitude", "longitude"]].values)
 fire_rad = np.radians(fire_df[["latitude", "longitude"]].values)

 # arbre des points feu
 tree = BallTree(fire_rad, metric="haversine")

# distance du plus proche voisin pour chaque point du grid
 dist, idx = tree.query(all_rad, k=1)

# seuil en km
 seuil_km = 1.8
 seuil_rad = seuil_km / 6371.0  # rayon Terre en km

# colonne in_fire
 grid_df[column_name] = (dist[:, 0] <= seuil_rad).astype(int)
 print("done")
 grid_df[f"{column_name}_distance"] = dist[:, 0] * 6371.0

 print(grid_df[column_name].value_counts())
 


In [54]:
Fire1.count()

longitude    2133
latitude     2133
dtype: int64

In [55]:
insert_fire_grid(Fire3,grid,"was_in_fire_2023")
insert_fire_grid(Fire2,grid,"was_in_fire_2022")    
insert_fire_grid(Fire1,grid,"was_in_fire_2021")

done
was_in_fire_2023
0    81861
1     7337
Name: count, dtype: int64
done
was_in_fire_2022
0    79457
1     9741
Name: count, dtype: int64
done
was_in_fire_2021
0    81006
1     8192
Name: count, dtype: int64


In [56]:
grid = grid.drop(columns=["landcover"])

In [57]:
grid.columns

Index(['latitude', 'longitude', 'was_in_fire_2023',
       'was_in_fire_2023_distance', 'was_in_fire_2022',
       'was_in_fire_2022_distance', 'was_in_fire_2021',
       'was_in_fire_2021_distance'],
      dtype='object')

In [58]:
grid.to_csv("grid_with_historical_fire.csv",index=False)